# 12.15 · LLM 评估 / LLM Evaluation

> **课程定位 / Where this fits**
> 第 15 课，**Part 12**。怎么知道一个 LLM(或它的生成)到底好不好?
> Lesson 15, **Part 12**. How do we know an LLM (or its output) is actually good?
>
> 评估生成式模型出了名地难——一个问题可能有**无数个都对的回答**, 没有唯一标准答案。本课覆盖三类常用评估: **困惑度(perplexity)** ——模型对文本有多"惊讶"(内在指标)；**BLEU / ROUGE** ——把生成结果和参考答案做 n-gram 重叠(翻译/摘要常用)；以及现代的 **LLM-as-judge**(用强模型当裁判)。我们**从零实现 perplexity、BLEU、ROUGE**, 理解每个指标量什么、有何坑。
> Evaluating generative models is notoriously hard — a question may have **countless valid answers**, no single ground truth. We cover three common families: **perplexity** — how "surprised" the model is by text (intrinsic); **BLEU / ROUGE** — n-gram overlap of generation vs reference (translation/summarization); and the modern **LLM-as-judge** (a strong model grades outputs). We **implement perplexity, BLEU, ROUGE from scratch**, understanding what each measures and its pitfalls.
>
> 💼 **实战/面试视角**："perplexity 是什么 / BLEU/ROUGE 区别 / 这些指标的局限 / LLM-as-judge / 基准污染" 是 LLM 评估常考。
> 💼 **Practical/interview angle:** "what is perplexity / BLEU vs ROUGE / their limits / LLM-as-judge / benchmark contamination" — common eval questions.

> 📐 **符号约定 / Notation**
> - 困惑度 perplexity $= \exp(\text{平均负对数似然})$ / exp of average negative log-likelihood
> - BLEU —— 精确率导向(precision) / precision-oriented
> - ROUGE —— 召回率导向(recall) / recall-oriented

> 💡 **面试相关 / Interview-relevant**
> - "困惑度的定义与直觉"（出镜率 ★★★★）
> - "BLEU(精确率) vs ROUGE(召回率)"（★★★★★）
> - "这些 n-gram 指标的根本局限"（★★★★，不懂语义/同义改写）
> - "LLM-as-judge 的优缺点"（★★★★）
> - "基准污染(data contamination)"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解为什么生成评估难、有哪几类指标。
   Understand why generation eval is hard and the metric families.
2. **从零实现困惑度**, 理解它衡量什么。
   Implement perplexity from scratch; understand what it measures.
3. **从零实现 BLEU 与 ROUGE**, 分清精确率vs召回率导向。
   Implement BLEU and ROUGE; distinguish precision vs recall orientation.
4. 了解 LLM-as-judge 与评估的坑。
   Know LLM-as-judge and evaluation pitfalls.

## 目录 / TOC
1. [为什么评估难 ⭐](#1)
2. [困惑度 Perplexity（从零）⭐](#2)
3. [BLEU 与 ROUGE（从零）⭐](#3)
4. [LLM-as-judge 与坑 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么评估难 ⭐ / Why Evaluation Is Hard

分类任务有明确的对错(准确率、F1)。但生成任务——翻译、摘要、对话——**同一个输入有无数个都很好的输出**。"今天天气好"可以翻成 "It's nice today" / "The weather is great today" / "Lovely weather today"……都对。怎么自动判分?
Classification has clear right/wrong (accuracy, F1). But generation — translation, summarization, dialogue — has **countless good outputs for one input**. "今天天气好" → "It's nice today" / "The weather is great today" / "Lovely weather today"… all valid. How to score automatically?

三类常用方法：
Three common families:
- **内在指标(intrinsic)**：不需要参考答案, 直接看模型对文本的概率——**困惑度(perplexity)**。衡量"语言建模能力"。
  **Intrinsic:** no reference needed, look at the model's probability of text — **perplexity**. Measures language-modeling ability.
- **基于参考(reference-based)**：拿生成结果和人写的**参考答案**比 n-gram 重叠——**BLEU(翻译)/ROUGE(摘要)**。快、自动, 但**不懂语义**。
  **Reference-based:** compare generation to a human **reference** by n-gram overlap — **BLEU (translation)/ROUGE (summarization)**. Fast, automatic, but **semantics-blind**.
- **模型/人工评判(judge)**：用**人**或**强 LLM**直接打分(质量、有用性、安全性)——**LLM-as-judge**。最贴近真实质量, 但贵/有偏。
  **Judge-based:** **humans** or a **strong LLM** score quality/helpfulness/safety — **LLM-as-judge**. Closest to real quality but costly/biased.


<a id="2"></a>
## 2. 困惑度 Perplexity（从零）⭐ / Perplexity From Scratch

**困惑度(perplexity)** 衡量一个语言模型对一段文本有多"惊讶/困惑"。直觉：好的语言模型应该觉得真实文本"很自然"(给它高概率)——困惑度就低。
**Perplexity** measures how "surprised/perplexed" a language model is by text. Intuition: a good LM finds real text "natural" (assigns high probability) → low perplexity.

定义：$\text{PPL} = \exp\big(-\frac{1}{N}\sum_i \log P(w_i \mid \text{context})\big)$ ——即**平均负对数似然取指数**。直观理解：**模型在每一步平均在多少个词里"纠结"**——PPL=10 意思是"平均像在 10 个等可能的词里猜"。**越低越好**。
Definition: $\text{PPL} = \exp\big(-\frac{1}{N}\sum_i \log P(w_i \mid \text{context})\big)$ — the **exp of average negative log-likelihood**. Intuition: roughly **how many words the model is "torn between" per step** — PPL=10 means "like guessing among 10 equally likely words." **Lower is better.**

下面对比三个语言模型的困惑度: **均匀(瞎猜)、unigram(只看词频)、bigram(看前一个词)** ——越聪明的模型困惑度越低。
Below we compare perplexity of three LMs: **uniform (random), unigram (word freq), bigram (uses previous word)** — smarter models have lower perplexity.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, re
from collections import Counter, defaultdict
import nltk; nltk.download("gutenberg", quiet=True); from nltk.corpus import gutenberg
sns.set_theme(style="whitegrid")

words = re.findall(r"[a-z]+", gutenberg.raw("austen-sense.txt").lower())
split = int(0.9*len(words)); train, test = words[:split], words[split:]
vocab = set(train); Vn = len(vocab)
uni = Counter(train); total = len(train)
bi = defaultdict(Counter)
for a, b in zip(train[:-1], train[1:]): bi[a][b] += 1

def perplexity(prob_fn, text):
    log_sum = 0.0
    for i, w in enumerate(text):
        prev = text[i-1] if i > 0 else None
        p = max(prob_fn(w, prev), 1e-10)                  # 防 log(0) / avoid log(0)
        log_sum += np.log(p)
    return float(np.exp(-log_sum / len(text)))            # exp(平均负对数似然) / exp(avg NLL)

p_uniform = lambda w, prev: 1.0 / Vn                                       # 均匀: 每个词等概率 / uniform
p_unigram = lambda w, prev: (uni[w] + 1) / (total + Vn)                    # unigram: 按词频(加平滑) / by word freq
def p_bigram(w, prev, lam=0.9):                                            # bigram: 线性插值回退到unigram / interpolation backoff
    # 真实n-gram模型用插值/回退(不是裸加1), 否则稀疏数据下bigram反而更差 / real LMs interpolate, else sparse hurts
    if prev is None or prev not in bi: return p_unigram(w, prev)
    bigram_mle = bi[prev][w] / sum(bi[prev].values())                      # 见过的bigram用MLE / seen-bigram MLE
    return lam * bigram_mle + (1 - lam) * p_unigram(w, prev)               # 插值: 见过用bigram, 没见过回退unigram

test_sample = test[:2000]
ppls = {"均匀(瞎猜)": perplexity(p_uniform, test_sample),
        "unigram(词频)": perplexity(p_unigram, test_sample),
        "bigram(看前一词)": perplexity(p_bigram, test_sample)}
fig, ax = plt.subplots(figsize=(6.5,4))
ax.bar(list(ppls.keys()), list(ppls.values()), color=["#bbb","#5a9","#39c"])
for i,(k,v) in enumerate(ppls.items()): ax.text(i, v, f"{v:.0f}", ha="center", va="bottom")
ax.set_ylabel("困惑度 (越低越好)"); ax.set_title("困惑度: 越聪明的语言模型, 困惑度越低")
plt.tight_layout(); plt.show()
for k, v in ppls.items(): print(f"  {k:18}: 困惑度 = {v:.1f}")
print(f"\n均匀模型≈词表大小{Vn}(纯瞎猜); unigram用词频降低; bigram用上下文进一步降低")
print("困惑度=模型对真实文本的'平均纠结词数', 越低=语言建模越好; GPT等就是用它衡量预训练")
print("⚠️ 局限: 只衡量'语言流畅度', 不衡量'回答是否正确/有用'; 不同分词/词表间不可直接比较")


<a id="3"></a>
## 3. BLEU 与 ROUGE（从零）⭐ / BLEU & ROUGE From Scratch

当有**参考答案**(人写的标准翻译/摘要)时，用 n-gram 重叠打分。两个经典指标方向相反(面试高频对比)：
With a **reference** (human translation/summary), score by n-gram overlap. Two classics with opposite orientations (high-frequency comparison):

- **BLEU(精确率导向, 翻译常用)**：**生成里的 n-gram 有多少出现在参考里**(生成的内容"准不准")。多个 n(1~4)的精确率几何平均, 再乘**简短惩罚(brevity penalty)** 防止用超短输出刷高精确率。
  **BLEU (precision-oriented, translation):** what fraction of the **generation's** n-grams appear in the reference (is the output "accurate"). Geometric mean of n-gram precisions (n=1..4) × a **brevity penalty** to stop gaming with too-short outputs.
- **ROUGE(召回率导向, 摘要常用)**：**参考里的 n-gram 有多少被生成覆盖**(参考的要点"漏没漏")。ROUGE-N 看 n-gram 召回, ROUGE-L 看**最长公共子序列(LCS)**。
  **ROUGE (recall-oriented, summarization):** what fraction of the **reference's** n-grams are covered by the generation (did we miss key points). ROUGE-N for n-gram recall, ROUGE-L for **longest common subsequence (LCS)**.

下面从零实现两者。
We implement both from scratch.


In [ ]:
def ngrams(tokens, n): return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def bleu(candidate, reference, max_n=4):
    cand, ref = candidate.split(), reference.split()
    precisions = []
    for n in range(1, max_n+1):
        cg, rg = ngrams(cand, n), ngrams(ref, n)
        if not cg: precisions.append(0); continue
        # 截断计数: 每个n-gram的命中不超过参考里的出现次数 / clipped count
        overlap = sum(min(c, rg[g]) for g, c in cg.items())
        precisions.append(overlap / max(sum(cg.values()), 1))
    if min(precisions) == 0: geo = 0.0
    else: geo = np.exp(np.mean([np.log(p) for p in precisions]))   # n-gram 精确率几何平均 / geometric mean
    bp = 1.0 if len(cand) > len(ref) else np.exp(1 - len(ref)/max(len(cand),1))   # 简短惩罚 / brevity penalty
    return bp * geo

def rouge_n(candidate, reference, n=1):
    cg, rg = ngrams(candidate.split(), n), ngrams(reference.split(), n)
    if not rg: return 0.0
    overlap = sum(min(cg[g], c) for g, c in rg.items())
    return overlap / sum(rg.values())                              # 召回: 参考n-gram被覆盖比例 / recall

def rouge_l(candidate, reference):
    a, b = candidate.split(), reference.split()
    dp = np.zeros((len(a)+1, len(b)+1), int)                       # LCS 动态规划(呼应11.9编辑距离) / LCS DP
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]
    if lcs == 0: return 0.0
    prec, rec = lcs/len(a), lcs/len(b); return 2*prec*rec/(prec+rec)   # LCS 的 F1 / F1 of LCS

reference = "the cat is sitting on the mat"
for cand in ["the cat is sitting on the mat",          # 完全一致 / identical
             "the cat sits on the mat",                # 同义改写 / paraphrase
             "a dog runs in the park"]:                # 完全不同 / unrelated
    print(f"候选: {cand!r}")
    print(f"  BLEU={bleu(cand,reference):.2f}  ROUGE-1={rouge_n(cand,reference,1):.2f}  ROUGE-L={rouge_l(cand,reference):.2f}")
print("\nBLEU(精确率): 生成的词组多少在参考里; ROUGE(召回率): 参考的词组多少被覆盖")
print("⚠️ 关键局限: 都只看词面重叠! '同义改写'(意思对但用词不同)会被打低分 → 这是n-gram指标的根本缺陷")


<a id="4"></a>
## 4. LLM-as-judge 与坑 + 小结 ⭐ / LLM-as-Judge & Pitfalls

BLEU/ROUGE 的根本问题(上面已见)：**只看词面、不懂语义**——一个意思完全对但换了说法的回答会被判低分；一个堆砌参考词但语无伦次的回答可能得高分。对开放式对话/推理, 它们几乎没用。
The core problem of BLEU/ROUGE (seen above): **surface-level, semantics-blind** — a correct paraphrase scores low; a word-salad echoing the reference may score high. For open-ended dialogue/reasoning, they're nearly useless.

**现代评估方法**：
**Modern evaluation:**
- **人工评估(human eval)**：金标准, 但贵、慢、主观(需多人+标注规范)。
  **Human eval:** gold standard, but costly, slow, subjective (needs multiple raters + guidelines).
- **LLM-as-judge**：用一个**强 LLM(如 GPT-4)** 当裁判, 给回答打分或在两个回答里选更好的。**快、便宜、和人类判断相关性高**, 现在极常用(如 MT-Bench、Chatbot Arena 的自动版)。**但有偏**：偏好更长的回答、偏好自己风格的输出(self-bias)、位置偏好(先看到的)等——要做去偏处理。
  **LLM-as-judge:** a **strong LLM (e.g. GPT-4)** scores answers or picks the better of two. **Fast, cheap, well-correlated with humans**, now very common (MT-Bench, auto Chatbot Arena). **But biased:** prefers longer answers, its own style (self-bias), position bias — needs debiasing.
- **基准测试(benchmarks)**：MMLU(知识)、HumanEval(代码)、GSM8K(数学)等标准题库, 有客观答案可自动判分。
  **Benchmarks:** MMLU (knowledge), HumanEval (code), GSM8K (math) — objective answers, auto-scorable.

**重要的坑(面试)：基准污染(data contamination)**。如果测试题**泄漏进了训练数据**(网上的题库被爬进预训练语料), 模型可能是"背过答案"而非真会——分数虚高。评估新模型时要警惕。
**Key pitfall: benchmark contamination.** If test questions **leaked into training data** (online benchmarks scraped into pretraining), the model may have "memorized answers" — inflated scores. Be wary when evaluating new models.

```
评估难: 生成有无数正确答案, 无唯一标准; 三类: 内在(perplexity)/基于参考(BLEU,ROUGE)/评判(human,LLM-judge)
困惑度: exp(平均负对数似然), =模型对文本'平均纠结词数', 越低语言建模越好; 但不衡量正确/有用
BLEU(精确率, 翻译): 生成n-gram多少在参考里 + 简短惩罚; ROUGE(召回率, 摘要): 参考n-gram多少被覆盖(ROUGE-L用LCS)
n-gram指标根本缺陷: 只看词面不懂语义, 同义改写被低估
LLM-as-judge: 强模型当裁判, 快/便宜/相关性高, 但有偏(长度/自我/位置偏好)需去偏
基准: MMLU/HumanEval/GSM8K; 坑=基准污染(测试题泄漏进训练→分数虚高)
```

### 💡 面试速查 / Interview cheat-sheet
1. **困惑度**: exp(平均NLL), 越低越好; 衡量语言建模, 不衡量正确性。
   Perplexity: exp(avg NLL), lower better; measures LM quality, not correctness.
2. **BLEU vs ROUGE**: BLEU精确率(翻译), ROUGE召回率(摘要); 都看n-gram重叠。
   BLEU vs ROUGE: BLEU precision (translation), ROUGE recall (summarization); both n-gram overlap.
3. **n-gram局限**: 只看词面, 同义改写被低估 → 不懂语义。
   n-gram limit: surface overlap, penalizes paraphrase → semantics-blind.
4. **LLM-as-judge**: 强模型打分, 快/相关性高, 但有偏(长度/位置/自我)。
   LLM-as-judge: strong model scores, fast/correlated, but biased.
5. **基准污染**: 测试题泄漏进训练→分数虚高, 评估要警惕。
   Contamination: test leaked into training → inflated scores; be wary.

### 下一节 / Next
**12.16 LLM 推理优化**——大模型生成很慢很贵(逐token串行+巨大)。本课讲让推理**又快又省**的关键工程: **KV cache、量化、投机解码、连续批处理(vLLM)** 等——这是把 LLM 真正部署上线的"最后一公里"。
**12.16 LLM Inference Optimization** — generation is slow and costly (serial per-token + huge models). We cover key engineering to make inference **fast and cheap**: **KV cache, quantization, speculative decoding, continuous batching (vLLM)** — the "last mile" of deploying LLMs.
